# RoBERTa-base SQuAD2 — DIMER extractive question answering tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/roberta-squad2-question-answering-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/roberta-squad2-question-answering-pipeline/blob/main/tutorials/roberta_question_answering_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-deepset%2Froberta--base--squad2-ffcc4d?style=flat)](https://huggingface.co/deepset/roberta-base-squad2) [![Upstream](https://img.shields.io/badge/Upstream-deepset--ai%2Fhaystack-181717?style=flat&logo=github&logoColor=white)](https://github.com/deepset-ai/haystack) [![arXiv](https://img.shields.io/badge/arXiv-1907.11692-b31b1b.svg)](https://arxiv.org/abs/1907.11692)

**Profile:** `TASK-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.1 — **standalone** (§3.6)  
**Capability:** extractive question answering with the SQuAD 2.0 unanswerable case using the pinned `deepset/roberta-base-squad2` weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/roberta_question_answering_pipeline/pipeline.py` at revision `3f6ba6704c32`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `adc3b06f79f797d1c575d5479d6f5efe54a9e3b4` (~498 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

`deepset/roberta-base-squad2` is the 125 M-parameter `roberta-base` encoder (Liu et al., 2019) with a span head, fine-tuned by deepset on SQuAD 2.0 — question-answer pairs **including unanswerable questions** — and published under **CC-BY-4.0** (attribution: deepset; the weights are redistributed unmodified). It is an **extractive reader**: it can only copy one contiguous span out of the passage you give it, and it can decide that the passage contains no answer. At inference the encoder reads `<s> question </s></s> context </s>` once and emits a start and an end logit per token; the carried module then applies the **upstream null-vs-span rule** (`DECISION_RULE`): softmax the start and end logits over the context tokens plus `<s>`, take the best span as the argmax of `P_start(i)·P_end(j)` with `i ≤ j < i + max_answer_tokens`, score "no answer" as `P_start(<s>)·P_end(<s>)`, and return the **empty answer** when the null score wins. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, retrieval or preprocessing fitting — the pinned checkpoint is used as published. What the upstream checkpoint supplies is the model and the byte-level BPE tokenizer; what the carried pipeline module adds is manifest verification, input validation with named ceilings, the decision rule, a fixed output contract, the SQuAD-style `exact_match` and `f1` helpers, and the `validate_inputs` and `evaluation_report` stage helpers.

**Snapshot note:** the pinned revision ships **no `tokenizer.json`** (a 7-file manifest), so `AutoTokenizer` builds `RobertaTokenizerFast` from `vocab.json` and `merges.txt`; the card-pass smoke printed no tokenizer warning. Section 3 stages and digest-verifies those seven files before the tokenizer or the model is constructed.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, author one passage and three questions (two answerable, one deliberately unanswerable) or upload your own, stage and digest-verify the immutable upstream snapshot, surface the pipeline's ceilings and the decision rule and validate the pairs into an input manifest before any model work, answer through the public API with an explicit `max_answer_tokens`, read `answer`, `score`, `no_answer_score`, `best_span_score` and `answerable` correctly (products of softmax masses, not calibrated probabilities), read from the machine-readable evaluation report what `exact_match`/`f1` on authored gold answers does and does not prove, and export every answer with its identifier plus provenance.

**This notebook does not demonstrate:** generative or abstractive answering, yes/no or counting questions, multi-hop reasoning across sentences, retrieval over many documents (this reader takes one passage the caller supplies), passages over `MAX_CONTEXT_TOKENS` (refused, not windowed — chunk them yourself), languages other than English, batching, top-k alternative answers, a tuned no-answer threshold offset, or any accuracy claim beyond a single-pair sample-sanity check. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available (float32 there too). CPU is adequate: the repository's model card records, for the Windows-venv smoke on an Intel Core Ultra 9 275HX, 4.41 s to load and digest-verify the 498 MB snapshot and 0.105 s / 0.032 s / 0.032 s for three `answer` calls on a 48-token passage. The pinned `torch==2.14.0` install and the 496 MB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python; what an encoder-only Transformer is; what span extraction means; why a reader can return a fluent-looking wrong span, or an empty answer for a question the passage does answer.
- **Data:** the default sample is **synthetic** — one two-sentence passage and three questions authored in code, with the author's own gold answers (the third question has no answer in the passage) — so nothing is downloaded and no private data is needed. Three pairs are a plumbing check, never an accuracy. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction; the expected file is a JSON list of `{"question", "context"}` objects, each optionally carrying `"answers"` (a list of acceptable strings; an empty list means unanswerable). Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded text remains in the notebook runtime; this pipeline does not send it to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `deepset/roberta-base-squad2` snapshot (~498 MB in total) at revision `adc3b06f79f7…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'transformers==4.57.6',
    'tokenizers==0.22.2',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
]
NOTEBOOK_SOURCE = {
    'repository': 'roberta-squad2-question-answering-pipeline',
    'repository_revision': '3f6ba6704c32a9a1484755d0c328290ca06d72b1',
    'embedded_module': 'src/roberta_question_answering_pipeline/pipeline.py',
    'embedded_modules': ['src/roberta_question_answering_pipeline/pipeline.py'],
    'module_sha256': '0749e184e4c56d7cd1609c5d687e904d1840ae99b9262ca4056c6215f2f1453e',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '1.1',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/roberta_question_answering_pipeline/` @ `3f6ba6704c32`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/roberta_question_answering_pipeline/pipeline.py`

In [ ]:
"""Extractive question answering with the pinned ``deepset/roberta-base-squad2`` checkpoint.

The class loads weights only from a digest-verified local snapshot (``weights/roberta-base-squad2/``) or,
when explicitly allowed, from the Hugging Face Hub at the pinned revision. One task method, ``answer``:
a question and a context in, one context span (or the SQuAD 2.0 empty "no answer") out, decided with
the same null-vs-span rule the upstream ``transformers`` question-answering pipeline applies.
"""

from __future__ import annotations

import hashlib
import json
import re
import string
from collections import Counter
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np

MODEL_ID = "deepset/roberta-base-squad2"
MODEL_REVISION = "adc3b06f79f797d1c575d5479d6f5efe54a9e3b4"
MODEL_LICENSE = "cc-by-4.0"
MODEL_KEY = "roberta-base-squad2"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Ceilings. Hard ceilings, no sliding window: a question/context pair that does not fit is rejected, never
# split or truncated. 64 and 384 are the upstream fine-tuning settings (`max_query_length=64`,
# `max_seq_len=386` with `doc_stride=128` in the pinned README); 64 + 384 + 4 special tokens = 452 <= the
# 512-token `model_max_length` in the pinned tokenizer_config.json, so an accepted pair fits in one pass.
MAX_QUESTION_TOKENS = 64
MAX_CONTEXT_TOKENS = 384
MAX_QUESTION_CHARS = 500  # pre-tokenisation guard; ~4 chars per BPE token on English text
MAX_CONTEXT_CHARS = 3_000
MAX_ANSWER_TOKENS = 64  # ceiling on the span length a caller may request
DEFAULT_MAX_ANSWER_TOKENS = 15  # the upstream pipeline's `max_answer_len` default
NULL_MASK_VALUE = -10000.0  # the upstream pipeline's fill for non-context positions before the softmax
DECISION_RULE = (
    "softmax start and end logits over the context tokens plus <s>; no_answer_score = P_start(<s>) * "
    "P_end(<s>); best span = argmax P_start(i) * P_end(j) over i <= j < i + max_answer_tokens inside the "
    "context; the empty answer is returned when no_answer_score > best span score (upstream "
    "handle_impossible_answer=True rule); scores are products of softmax masses, not calibrated probabilities"
)


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _read_manifest(root: Path) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        return json.load(fh)


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one (question, context) pair of non-empty str; the answer is a span of the context or empty",
    "question_chars": [1, MAX_QUESTION_CHARS],
    "context_chars": [1, MAX_CONTEXT_CHARS],
    "question_tokens": [1, MAX_QUESTION_TOKENS],
    "context_tokens": [1, MAX_CONTEXT_TOKENS],
    "max_answer_tokens": [1, MAX_ANSWER_TOKENS],
    "decision_rule": DECISION_RULE,
    "preprocessing": (
        "byte-level BPE (vocab.json + merges.txt, no lower-casing) of the pair "
        "<s> question </s></s> context </s>; hard ceilings, no sliding window: a question over "
        "MAX_QUESTION_TOKENS or a context over MAX_CONTEXT_TOKENS is rejected with a ValueError naming the "
        "count, never truncated or chunked"
    ),
}


def _check_text(text: Any, name: str, max_chars: int) -> str:
    if not isinstance(text, str):
        raise TypeError(f"{name} must be str, got {type(text).__name__}")
    if not text.strip():
        raise ValueError(f"{name} is empty")
    if len(text) > max_chars:
        raise ValueError(f"{name} has {len(text)} chars; ceiling is MAX_{name.upper()}_CHARS={max_chars}")
    return text


def _check_inputs(question: Any, context: Any, max_answer_tokens: Any) -> tuple[str, str]:
    """Raise TypeError/ValueError naming the first violated ceiling; return (question, context).

    The token ceilings are not checked here because they need the loaded tokenizer;
    ``_check_token_counts`` applies them inside the pipeline once the counts are known.
    """
    question = _check_text(question, "question", MAX_QUESTION_CHARS)
    context = _check_text(context, "context", MAX_CONTEXT_CHARS)
    if isinstance(max_answer_tokens, bool) or not isinstance(max_answer_tokens, int):
        raise TypeError("max_answer_tokens must be an int")
    if not 1 <= max_answer_tokens <= MAX_ANSWER_TOKENS:
        raise ValueError(
            f"max_answer_tokens must be between 1 and {MAX_ANSWER_TOKENS}, got {max_answer_tokens}"
        )
    return question, context


def _check_token_counts(n_question: int, n_context: int) -> tuple[int, int]:
    """The token ceilings, applied once the tokenizer has counted (no special tokens)."""
    if n_question > MAX_QUESTION_TOKENS:
        raise ValueError(
            f"question is {n_question} tokens; ceiling is MAX_QUESTION_TOKENS={MAX_QUESTION_TOKENS}"
        )
    if n_context > MAX_CONTEXT_TOKENS:
        raise ValueError(f"context is {n_context} tokens; ceiling is MAX_CONTEXT_TOKENS={MAX_CONTEXT_TOKENS}")
    return n_question, n_context


def validate_inputs(
    questions: Sequence[str],
    contexts: Sequence[str],
    *,
    max_answer_tokens: int = DEFAULT_MAX_ANSWER_TOKENS,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-pair observations, verdict).

    Rejection is reported by raising exactly as ``answer`` would: both route through ``_check_inputs``.
    ``answer`` takes one pair per call, so ``questions``/``contexts`` are the parallel batch the notebook
    will loop over. The token ceilings (``MAX_QUESTION_TOKENS``, ``MAX_CONTEXT_TOKENS``) need the loaded
    tokenizer and are enforced inside ``answer``, which reports both counts in every result.
    """
    for label, value in (("questions", questions), ("contexts", contexts)):
        if isinstance(value, str | bytes) or not isinstance(value, Sequence):
            raise TypeError(f"{label} must be a sequence of str, not a single string")
    if not questions:
        raise ValueError("questions must hold at least one item")
    if len(questions) != len(contexts):
        raise ValueError("questions and contexts must have the same length")
    checked = [_check_inputs(q, c, max_answer_tokens) for q, c in zip(questions, contexts, strict=True)]
    if names is not None and len(names) != len(checked):
        raise ValueError("names must have one entry per pair")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {"id": names[i] if names else f"pair{i:02d}", "question_chars": len(q), "context_chars": len(c)}
            for i, (q, c) in enumerate(checked)
        ],
        "max_answer_tokens": max_answer_tokens,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def normalize_answer(text: str) -> str:
    """SQuAD-style normalisation: lower-case, drop punctuation and the articles a/an/the, collapse spaces."""
    text = "".join(ch for ch in text.lower() if ch not in set(string.punctuation))
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())


def exact_match(prediction: str, gold_answers: Sequence[str]) -> float:
    """1.0 if the normalised prediction equals any normalised gold answer (an empty gold list means
    unanswerable and matches only the empty prediction), else 0.0."""
    golds = [normalize_answer(g) for g in gold_answers] or [""]
    return 1.0 if normalize_answer(prediction) in golds else 0.0


def f1(prediction: str, gold_answers: Sequence[str]) -> float:
    """Best token-overlap F1 against any gold answer, SQuAD 2.0 style: for an unanswerable gold (empty
    list or empty string) the score is 1.0 only when the prediction is empty too."""
    golds = list(gold_answers) or [""]
    pred_tokens = normalize_answer(prediction).split()
    best = 0.0
    for gold in golds:
        gold_tokens = normalize_answer(gold).split()
        if not pred_tokens or not gold_tokens:
            score = float(pred_tokens == gold_tokens)
        else:
            common = sum((Counter(pred_tokens) & Counter(gold_tokens)).values())
            if common == 0:
                score = 0.0
            else:
                precision, recall = common / len(pred_tokens), common / len(gold_tokens)
                score = 2 * precision * recall / (precision + recall)
        best = max(best, score)
    return best


def evaluation_report(
    result: Mapping[str, Any], gold_answers: Sequence[str] | None = None, *, sample_kind: str = "synthetic"
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report for one ``answer`` result.

    With ``gold_answers`` (the SQuAD convention: a list of acceptable strings; an empty list means the
    question is unanswerable) the report carries ``exact_match`` and ``f1`` for that single pair and the
    verdict ``sample-sanity`` — one pair is a plumbing check, not an accuracy. Without gold the verdict is
    ``not-measurable``.
    """
    prediction = str(result.get("answer", ""))
    supplied = gold_answers is not None
    metrics = []
    if supplied:
        metrics = [
            {
                "id": "exact_match",
                "value": exact_match(prediction, gold_answers),
                "estimation": "single pair",
            },
            {"id": "f1", "value": f1(prediction, gold_answers), "estimation": "single pair"},
        ]
    return {
        "task": "extractive question answering with the SQuAD 2.0 unanswerable case",
        "decision_rule": DECISION_RULE,
        "sample_kind": sample_kind,
        "n_pairs": 1,
        "answerable": bool(prediction),
        "metrics": metrics,
        "baselines": [],
        "verdict": "sample-sanity" if supplied else "not-measurable",
        "reason": (
            "exact_match and f1 are computed for one (question, context, gold) triple with the repository's "
            "SQuAD-style normalisation; a single pair states no dispersion and is not an accuracy"
            if supplied
            else "no gold answer was supplied, so exact_match and f1 cannot be computed"
        ),
        "needs": (
            "gold answer spans (SQuAD 2.0 format: a list of acceptable strings per question, empty for "
            "unanswerable) over enough held-out pairs from the deployment domain to state a dispersion, "
            "scored with the repository's exact_match and f1 helpers"
        ),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


@dataclass
class RoBERTaQuestionAnsweringPipeline:
    """``_runner(question, context)`` -> ``(start_logits (T,), end_logits (T,), offsets (T, 2),
    context_mask (T,))`` for the encoded pair; ``_count_tokens(text)`` -> BPE token count without special
    tokens. Both injectable so tests run offline."""

    _runner: Callable[[str, str], tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]]
    _count_tokens: Callable[[str], int]
    device: str = "cpu"
    source: str = "injected"

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> RoBERTaQuestionAnsweringPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, kwargs, source = str(root), dict(local_files_only=True), "local-snapshot"
        elif allow_download:
            location, kwargs, source = MODEL_ID, dict(revision=MODEL_REVISION), "hf-hub"
        else:
            raise FileNotFoundError(f"no verified snapshot at {root} and allow_download=False")
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import AutoTokenizer, RobertaForQuestionAnswering

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        tokenizer = AutoTokenizer.from_pretrained(location, trust_remote_code=False, **kwargs)
        model = RobertaForQuestionAnswering.from_pretrained(
            location, dtype=torch.float32, trust_remote_code=False, **kwargs
        )
        model = model.to(resolved_device).eval()

        def count_tokens(text: str) -> int:
            return len(tokenizer(text, add_special_tokens=False, truncation=False)["input_ids"])

        def runner(question: str, context: str) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
            enc = tokenizer(
                question, context, return_tensors="pt", return_offsets_mapping=True, truncation=False
            )
            offsets = enc.pop("offset_mapping")[0].numpy()
            context_mask = np.array([sid == 1 for sid in enc.sequence_ids(0)], dtype=bool)
            with torch.inference_mode():
                out = model(**enc.to(resolved_device))
            return (
                out.start_logits[0].float().cpu().numpy(),
                out.end_logits[0].float().cpu().numpy(),
                offsets,
                context_mask,
            )

        return cls(runner, count_tokens, resolved_device, source)

    def _validate(self, question: Any, context: Any, max_answer_tokens: Any) -> tuple[int, int]:
        question, context = _check_inputs(question, context, max_answer_tokens)
        return _check_token_counts(self._count_tokens(question), self._count_tokens(context))

    def answer(
        self, question: str, context: str, *, max_answer_tokens: int = DEFAULT_MAX_ANSWER_TOKENS
    ) -> dict[str, Any]:
        """Extract the best context span or the SQuAD 2.0 empty answer (upstream null-vs-span rule)."""
        n_question, n_context = self._validate(question, context, max_answer_tokens)
        start, end, offsets, context_mask = self._runner(question, context)
        start, end = np.asarray(start, dtype=np.float64), np.asarray(end, dtype=np.float64)
        context_mask = np.asarray(context_mask, dtype=bool)
        offsets = np.asarray(offsets)
        if not (start.shape == end.shape == context_mask.shape) or offsets.shape != (start.shape[0], 2):
            raise RuntimeError(
                f"runner returned inconsistent shapes {start.shape} {end.shape} {offsets.shape}"
            )
        allowed = context_mask.copy()
        allowed[0] = True  # <s> stays eligible so the null answer has a score
        start = np.where(allowed, start, NULL_MASK_VALUE)
        end = np.where(allowed, end, NULL_MASK_VALUE)
        p_start = np.exp(start - start.max())
        p_start /= p_start.sum()
        p_end = np.exp(end - end.max())
        p_end /= p_end.sum()
        no_answer_score = float(p_start[0] * p_end[0])
        p_start[0] = p_end[0] = 0.0
        outer = np.tril(np.triu(np.outer(p_start, p_end)), max_answer_tokens - 1)
        outer[~context_mask, :] = 0.0
        outer[:, ~context_mask] = 0.0
        best_index = int(np.argmax(outer))
        span_start, span_end = np.unravel_index(best_index, outer.shape)
        best_span_score = float(outer[span_start, span_end])
        if no_answer_score > best_span_score:
            answer_text, char_start, char_end, score = "", 0, 0, no_answer_score
        else:
            char_start, char_end = int(offsets[span_start][0]), int(offsets[span_end][1])
            answer_text, score = context[char_start:char_end], best_span_score
        return {
            "answer": answer_text,
            "score": score,
            "start": char_start,
            "end": char_end,
            "no_answer_score": no_answer_score,
            "best_span_score": best_span_score,
            "answerable": bool(answer_text),
            "question_tokens": n_question,
            "context_tokens": n_context,
            "max_answer_tokens": max_answer_tokens,
            "decision_rule": DECISION_RULE,
            "device": self.device,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `7`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `adc3b06f79f7…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `RoBERTaQuestionAnsweringPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "roberta-base-squad2",
  "modelId": "deepset/roberta-base-squad2",
  "revision": "adc3b06f79f797d1c575d5479d6f5efe54a9e3b4",
  "files": [
    {
      "path": "README.md",
      "bytes": 9181,
      "sha256": "04834cd9007b2cdd1b1bdd501b571579577331e467be6e24e47d40f060558a29"
    },
    {
      "path": "config.json",
      "bytes": 571,
      "sha256": "64fa58495a722d57609c22f199824bfe98c19be068136a70c268214a08cb8060"
    },
    {
      "path": "merges.txt",
      "bytes": 456318,
      "sha256": "1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5"
    },
    {
      "path": "model.safetensors",
      "bytes": 496254442,
      "sha256": "ac5db66fdcfecb400345d09787b71009d60805ef9883451071669cf951b5e2c7"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 772,
      "sha256": "c611b1f7d416eb001ee4f293d903ea8c88e703463f1d403f1866a0352743fd00"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 79,
      "sha256": "7a33226d4265e3989cc6341666af179d0cc710136f4059aae0dd8c0797cba556"
    },
    {
      "path": "vocab.json",
      "bytes": 898822,
      "sha256": "06b4d46c8e752d410213d9548eb27a54db70fda0319b6271fb8d59dead5e1cab"
    }
  ],
  "totalBytes": 497620185
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = RoBERTaQuestionAnsweringPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Author the synthetic sample or optional BYOD

The default sample is **synthetic**: one two-sentence passage about the Eiffel Tower and three questions authored in this cell, the same passage the repository's card-pass smoke used. Two questions are answered literally in the passage and carry the author's gold spans; the third (`What colour is the tower painted?`) is **deliberately unanswerable** and carries an empty gold list, the SQuAD 2.0 convention. The card's smoke observations — `Gustave Eiffel`, `1887 to 1889`, and the empty answer — are one run on one machine, not expected values this notebook asserts. The sample identity and a SHA-256 of its text are printed so an export can be tied to exactly these inputs.

One Colab form parameter fixes the span-length cap for every call: `ANSWER_MAX_TOKENS` (default 15, the package's `DEFAULT_MAX_ANSWER_TOKENS` and the upstream pipeline's `max_answer_len` default). It is checked against the carried module's ceiling in the next section.

BYOD is optional and disabled by default. Expected BYOD input: one UTF-8 JSON file holding a list of objects with `question` and `context` strings and, optionally, `answers` (a list of acceptable gold strings; `[]` for unanswerable). Each question must be at most `MAX_QUESTION_CHARS` characters and `MAX_QUESTION_TOKENS` BPE tokens, each context at most `MAX_CONTEXT_CHARS` characters and `MAX_CONTEXT_TOKENS` tokens, which the pipeline enforces by rejecting, not by truncating or windowing — chunk long documents yourself. The upload stays inside this runtime.

In [ ]:
import hashlib

USE_BYOD = False  # @param {type:"boolean"}
ANSWER_MAX_TOKENS = 15  # @param {type:"integer"}

if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    sample_name = next(iter(uploaded))
    records = json.loads(uploaded[sample_name].decode('utf-8'))
    if not isinstance(records, list) or not records:
        raise ValueError(f'{sample_name}: expected a non-empty JSON list of {{question, context[, answers]}} objects')
    questions = [str(r['question']) for r in records]
    contexts = [str(r['context']) for r in records]
    golds = [list(r['answers']) if 'answers' in r else None for r in records]
    sample_kind = 'BYOD upload'
else:
    passage = (
        'The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. '
        'It is named after the engineer Gustave Eiffel, whose company designed and built the tower from 1887 to 1889.'
    )
    questions = [
        'Who designed the Eiffel Tower?',
        'When was the tower built?',
        'What colour is the tower painted?',
    ]
    contexts = [passage] * len(questions)
    golds = [['Gustave Eiffel'], ['1887 to 1889', 'from 1887 to 1889'], []]
    sample_name = 'synthetic_eiffel_tower_qa'
    sample_kind = 'synthetic (authored in this cell)'
item_ids = [f'pair{index:02d}' for index in range(len(questions))]
sample_sha256 = hashlib.sha256('\n'.join(q + '\t' + c for q, c in zip(questions, contexts, strict=True)).encode('utf-8')).hexdigest()
print({'sample': sample_name, 'sample_kind': sample_kind, 'pairs': len(questions), 'text_sha256': sample_sha256, 'max_answer_tokens': ANSWER_MAX_TOKENS, 'with_gold': sum(g is not None for g in golds)})
for item_id, question, gold in zip(item_ids, questions, golds, strict=True):
    print(f'{item_id}: {question}  gold={gold}')

## 5. Validate the inputs → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `answer` applies — both route through the same private `_check_inputs` — so the text types, non-emptiness, the character ceilings `MAX_QUESTION_CHARS`/`MAX_CONTEXT_CHARS` and `max_answer_tokens` in 1..`MAX_ANSWER_TOKENS` are enforced identically. `answer` takes one pair per call, so the helper validates the whole parallel batch the notebook will loop over and returns one **input manifest** naming the schema and ceilings, each pair's identifier and character counts, the setting in force, and the verdict; it is written to `outputs/roberta_question_answering_input_manifest.json`. The token ceilings `MAX_QUESTION_TOKENS` (64) and `MAX_CONTEXT_TOKENS` (384, the upstream fine-tuning window) need the real tokenizer and are therefore enforced inside `answer`, which **rejects with a `ValueError` naming the count, never silently truncates or windows**; every result reports both counts. `DECISION_RULE` states the null-vs-span rule in force. To show what rejection looks like, the cell also validates an out-of-range `max_answer_tokens` and records the pipeline's own error message as a finding. Nothing here trims or alters the texts.

In [ ]:
import json

os.makedirs('outputs', exist_ok=True)
ceilings = {'MAX_QUESTION_CHARS': MAX_QUESTION_CHARS, 'MAX_CONTEXT_CHARS': MAX_CONTEXT_CHARS, 'MAX_QUESTION_TOKENS': MAX_QUESTION_TOKENS, 'MAX_CONTEXT_TOKENS': MAX_CONTEXT_TOKENS, 'MAX_ANSWER_TOKENS': MAX_ANSWER_TOKENS, 'DEFAULT_MAX_ANSWER_TOKENS': DEFAULT_MAX_ANSWER_TOKENS}
print(ceilings)
print({'decision_rule': DECISION_RULE})
input_manifest = validate_inputs(questions, contexts, max_answer_tokens=ANSWER_MAX_TOKENS, names=item_ids)
# Demonstrate rejection on a setting that breaks a ceiling; the finding is recorded, not swallowed.
try:
    validate_inputs(questions, contexts, max_answer_tokens=MAX_ANSWER_TOKENS + 1)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'max-answer-tokens-ceiling-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/roberta_question_answering_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))
print({'token_ceilings': 'enforced by answer() with the real tokenizer; reported as question_tokens / context_tokens'})

## 6. Answer and read the outputs correctly

`answer(question, context, max_answer_tokens=...)` runs one pair through the encoder and returns a dict: `answer` (the extracted span, or `''` for the SQuAD 2.0 no-answer), `start`/`end` (character offsets into `context`, both `0` for no-answer), `score` (the score of what was returned), `no_answer_score` (`P_start(<s>)·P_end(<s>)`), `best_span_score` (the best span's score even when the null won), `answerable` (whether a span was returned), `question_tokens`/`context_tokens` (the counts checked against the ceilings), `max_answer_tokens`, `decision_rule`, `device`, `source` and the model identity. **Score semantics:** every score is a **product of two softmax masses** normalised over the passage — a ranking signal that shrinks as the passage grows, **not a calibrated probability** — and the only decision the pipeline ships is the null comparison; there is no shipped no-answer threshold offset, so a caller who needs to trade missed facts against fabricated ones owns that margin, judged on their own labelled pairs. The run is deterministic for a given pair, setting, weights, device and library versions (no sampling, `model.eval()`, no seed needed); float32 kernel differences between CPU and CUDA can move a score in the last digits and flip a near-tied span or null decision. The checks below are falsifiable plumbing checks — one result per pair, every span a literal substring of its passage at the reported offsets, every count within its ceiling — plus a per-call wall time measured on the runtime identified in Section 1 (the first call includes warm-up). Look for a name for `pair00`, a date range for `pair01`, and an empty answer for `pair02`; whether they are *right* is what Section 7 checks against the authored gold, for three pairs only.

In [ ]:
import time

results = []
for item_id, question, context in zip(item_ids, questions, contexts, strict=True):
    started = time.perf_counter()
    result = pipe.answer(question, context, max_answer_tokens=ANSWER_MAX_TOKENS)
    elapsed = time.perf_counter() - started
    results.append({'id': item_id, 'question': question, 'context': context, 'seconds': round(elapsed, 3), **result})
    print(f"{item_id} [{'answerable' if result['answerable'] else 'no answer'}] score={result['score']:.4f} null={result['no_answer_score']:.4f} best_span={result['best_span_score']:.4f} {result['question_tokens']}+{result['context_tokens']} tokens, {elapsed:.2f} s")
    print(f"    {result['answer']!r} [{result['start']}:{result['end']}]")
checks = {
    'one_result_per_pair': len(results) == len(questions),
    'span_is_substring_at_offsets': all(r['context'][r['start']:r['end']] == r['answer'] for r in results),
    'question_within_ceiling': all(r['question_tokens'] <= MAX_QUESTION_TOKENS for r in results),
    'context_within_ceiling': all(r['context_tokens'] <= MAX_CONTEXT_TOKENS for r in results),
    'scores_in_unit_interval': all(0.0 <= r[k] <= 1.0 for r in results for k in ('score', 'no_answer_score', 'best_span_score')),
    'null_rule_consistent': all(r['answerable'] == (r['no_answer_score'] <= r['best_span_score']) for r in results),
    'setting_echoed': all(r['max_answer_tokens'] == ANSWER_MAX_TOKENS for r in results),
}
if not all(checks.values()):
    raise RuntimeError(f'answer output failed a sanity check: {checks}')
print({'checks': checks, 'decision_rule': results[0]['decision_rule'], 'unanswered': [r['id'] for r in results if not r['answerable']]})

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. The repository ships the two official SQuAD 2.0 measures as helpers — `exact_match` (the normalised prediction equals a normalised gold answer; lower-cased, punctuation and the articles a/an/the removed) and `f1` (best token-overlap F1 against any gold answer; an empty gold list means unanswerable and scores 1.0 only for an empty prediction) — and the report carries both **for one pair** with the verdict `sample-sanity`: three authored pairs are a plumbing check of the decision rule, **not an accuracy**, and the helper says so in `reason`. Without gold the verdict is `not-measurable` and `needs` names the held-out gold spans a real evaluation requires. The cell writes the report for the first pair to `outputs/roberta_question_answering_evaluation_report.json` and prints the per-pair scores for all of them. The upstream dev-set figures in the model card (exact 79.87 / F1 82.91 on 11,873 SQuAD 2.0 questions) are upstream claims, not measured here.

In [ ]:
reports = [evaluation_report(r, gold, sample_kind=sample_kind) for r, gold in zip(results, golds, strict=True)]
report = reports[0]
with open('outputs/roberta_question_answering_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2))
per_pair = [{'id': r['id'], 'verdict': rep['verdict'], **{m['id']: m['value'] for m in rep['metrics']}} for r, rep in zip(results, reports, strict=True)]
print({'per_pair': per_pair})
if report['verdict'] == 'not-measurable':
    print('No metric is reported: no gold answers were supplied; score your own held-out pairs with exact_match and f1.')
else:
    print('sample-sanity only: exact_match/f1 on a handful of authored pairs is a plumbing check, not an accuracy.')

## 8. Export the answers and provenance

Two further files are written under `outputs/` beside the input manifest and the evaluation report: `outputs/roberta_question_answering_answers.csv` — one row per pair with its identifier, the question, the extracted answer, its offsets, the three scores, `answerable`, both token counts and the wall time, so every answer maps back to its pair — and `outputs/roberta_question_answering_result.json`, which carries the same items plus each pair's context and gold, the per-pair evaluation reports, the setting in force, the ceilings, the sanity checks, the input manifest, the sample identity and digest, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence (CC-BY-4.0, attribution deepset), the verified snapshot summary, and the runtime identity (Python, `torch`, `transformers`, device, dtype). No credentials are involved in any step, so none can reach the export.

In [ ]:
import csv

items = [
    {'id': r['id'], 'question': r['question'], 'answer': r['answer'], 'start': r['start'], 'end': r['end'], 'score': r['score'], 'no_answer_score': r['no_answer_score'], 'best_span_score': r['best_span_score'], 'answerable': r['answerable'], 'question_tokens': r['question_tokens'], 'context_tokens': r['context_tokens'], 'seconds': r['seconds']}
    for r in results
]
with open('outputs/roberta_question_answering_answers.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(items[0]))
    writer.writeheader()
    writer.writerows(items)
payload = {
    'items': [{**item, 'context': r['context'], 'gold': gold} for item, r, gold in zip(items, results, golds, strict=True)],
    'evaluation_reports': reports,
    'max_answer_tokens': ANSWER_MAX_TOKENS,
    'decision_rule': DECISION_RULE,
    'ceilings': ceilings,
    'sanity_checks': checks,
    'answers_file': 'outputs/roberta_question_answering_answers.csv',
    'input_manifest': input_manifest,
    'evaluation_report': report,
    'sample': {'name': sample_name, 'kind': sample_kind, 'pairs': len(questions), 'text_sha256': sample_sha256},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'model_attribution': 'deepset (https://huggingface.co/deepset/roberta-base-squad2), weights redistributed unmodified under CC-BY-4.0',
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': len(snapshot['files']), 'total_bytes': snapshot.get('totalBytes'), 'fetched_this_run': fetched},
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
        'dtype': 'float32',
        'source': pipe.source,
    },
}
with open('outputs/roberta_question_answering_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The returned spans are the model's highest-scoring contiguous substrings of *your* passage under the null-vs-span rule: a span can be fluent, plausible and wrong, and an empty answer can suppress a fact the passage states in other words. Every score is a product of softmax masses, not a calibrated probability, and no no-answer margin is shipped — `no_answer_score` and `best_span_score` are reported so you can set one on your own labelled pairs. On the synthetic sample the checks prove only that the input contract, the verified snapshot load, the decision rule and the offset bookkeeping work end to end; the `sample-sanity` `exact_match`/`f1` on three authored pairs is a plumbing check, and a real evaluation needs held-out gold spans from your own domain over enough pairs to state a dispersion, read separately for answerable and unanswerable questions. Passages above `MAX_CONTEXT_TOKENS` are refused rather than windowed, so a long document must be chunked by the caller and an answer that straddles a chunk boundary is lost. The pipeline exposes no retrieval, batching, top-k alternatives or generative answering. Inference is deterministic on a fixed device and dtype, but CPU and CUDA float32 kernels can diverge on a near-tied span.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model snapshot, validate the demonstrated inputs against the enforced ceilings, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, reading-comprehension accuracy on any domain, a usable no-answer threshold, safety for high-consequence decisions, or production fitness on an unseen domain.

**Troubleshooting.** `RuntimeError: Core dependencies changed while older modules were loaded` in Section 1: the pinned install replaced a package the runtime had pre-imported — restart the runtime and rerun from the top. `FileNotFoundError: snapshot file missing` or a `sha256`/`size` `ValueError` in Section 3: a staged file is incomplete or altered — delete it from `weights/roberta-base-squad2/` and rerun Section 3. `ValueError: context is N tokens; ceiling is MAX_CONTEXT_TOKENS=384` in Section 6: split that BYOD passage into shorter chunks and rerun from Section 4. An empty answer for a question you believe the passage answers: the null score won — compare `no_answer_score` with `best_span_score` in the export, and try rephrasing the question closer to the passage's wording. A span cut short: raise `ANSWER_MAX_TOKENS` (ceiling `MAX_ANSWER_TOKENS`) and rerun Section 6.

**Next experiments.** Rephrase the unanswerable question so it *is* answerable from the passage and watch the null score fall; append an unrelated paragraph to the passage and watch every score shrink as the softmax mass spreads; upload a small BYOD file with gold answers from your own documents and read the per-pair `exact_match`/`f1` — the first step towards the real evaluation the report asks for; run the same pairs on a CUDA runtime and diff the scores against the CPU run. None of these turns the sample result into evidence of production fitness.

## References

- Repository README: https://github.com/kurtvalcorza/roberta-squad2-question-answering-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/roberta-squad2-question-answering-pipeline/blob/main/MODEL_CARD.md
- Weight provenance and CC-BY-4.0 attribution: https://github.com/kurtvalcorza/roberta-squad2-question-answering-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model (deepset, CC-BY-4.0): https://huggingface.co/deepset/roberta-base-squad2
- Upstream code: https://github.com/deepset-ai/haystack
- RoBERTa: A Robustly Optimized BERT Pretraining Approach (Liu et al., 2019): https://arxiv.org/abs/1907.11692
- Know What You Don't Know: Unanswerable Questions for SQuAD (Rajpurkar et al., ACL 2018): https://arxiv.org/abs/1806.03822